In [114]:
import numpy as np
import pandas as pd
import seaborn as sns

In [115]:
df = pd.read_csv("health_lifestyle_dataset.csv")

In [116]:
df.head()

,id,age,gender,bmi,daily_steps,sleep_hours,water_intake_l,calories_consumed,smoker,alcohol,resting_hr,systolic_bp,diastolic_bp,cholesterol,family_history,disease_risk
0,1,56,Male,20.5,4198,3.9,3.4,1602,0,0,97,161,111,240,0,0
1,2,69,Female,33.3,14359,9.0,4.7,2346,0,1,68,116,65,207,0,0
2,3,46,Male,31.6,1817,6.6,4.2,1643,0,1,90,123,99,296,0,0
3,4,32,Female,38.2,15772,3.6,2.0,2460,0,0,71,165,95,175,0,0
4,5,60,Female,33.6,6037,3.8,4.0,3756,0,1,98,139,61,294,0,0


In [117]:
df.shape

(100000, 16)

In [118]:
from re import L
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['gender'] = le.fit_transform(df['gender'])
df = df.astype(int)

In [119]:
df

,id,age,gender,bmi,daily_steps,sleep_hours,water_intake_l,calories_consumed,smoker,alcohol,resting_hr,systolic_bp,diastolic_bp,cholesterol,family_history,disease_risk
0,1,56,1,20,4198,3,3,1602,0,0,97,161,111,240,0,0
1,2,69,0,33,14359,9,4,2346,0,1,68,116,65,207,0,0
2,3,46,1,31,1817,6,4,1643,0,1,90,123,99,296,0,0
3,4,32,0,38,15772,3,2,2460,0,0,71,165,95,175,0,0
4,5,60,0,33,6037,3,4,3756,0,1,98,139,61,294,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,53,1,33,4726,3,2,3118,0,1,56,105,76,282,0,0
99996,99997,22,1,35,11554,4,3,1967,0,0,51,149,77,192,0,0
99997,99998,37,1,18,3924,3,1,2328,0,0,69,92,117,218,0,0
99998,99999,72,0,27,16110,5,0,3093,0,0,93,164,72,188,0,0


In [120]:
df.isnull().sum()

,0
id,0
age,0
gender,0
bmi,0
daily_steps,0
sleep_hours,0
water_intake_l,0
calories_consumed,0
smoker,0
alcohol,0


In [121]:
df.duplicated().sum()

np.int64(0)

In [122]:
df = df.drop(columns=['id'])

In [123]:
df

,age,gender,bmi,daily_steps,sleep_hours,water_intake_l,calories_consumed,smoker,alcohol,resting_hr,systolic_bp,diastolic_bp,cholesterol,family_history,disease_risk
0,56,1,20,4198,3,3,1602,0,0,97,161,111,240,0,0
1,69,0,33,14359,9,4,2346,0,1,68,116,65,207,0,0
2,46,1,31,1817,6,4,1643,0,1,90,123,99,296,0,0
3,32,0,38,15772,3,2,2460,0,0,71,165,95,175,0,0
4,60,0,33,6037,3,4,3756,0,1,98,139,61,294,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,53,1,33,4726,3,2,3118,0,1,56,105,76,282,0,0
99996,22,1,35,11554,4,3,1967,0,0,51,149,77,192,0,0
99997,37,1,18,3924,3,1,2328,0,0,69,92,117,218,0,0
99998,72,0,27,16110,5,0,3093,0,0,93,164,72,188,0,0


In [124]:
X = df.drop(columns=['disease_risk'],axis = 1)
y = df['disease_risk']

In [125]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [138]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42
)

In [139]:
dt.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [140]:
y_pred = dt.predict(X_test)

In [141]:
from sklearn.metrics import accuracy_score
accuracy_score = accuracy_score(y_test, y_pred)
accuracy_score

0.6257272727272727

In [157]:
from sklearn.model_selection import RandomizedSearchCV
param_grid = {
    'max_depth': [5,7,10,15],
    'min_samples_split': [5,10,20],
    'min_samples_leaf': [2,4,8],
    'criterion': ['gini','entropy']
}

In [158]:
rand_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_grid,
    cv=5,
    n_iter=10,
    scoring='f1',
    n_jobs=-1,
    random_state=42)
rand_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5,
                   estimator=DecisionTreeClassifier(class_weight='balanced',
                                                    random_state=42),
                   n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy'],
                                        'max_depth': [5, 7, 10, 15],
                                        'min_samples_leaf': [2, 4, 8],
                                        'min_samples_split': [5, 10, 20]},
                   random_state=42, scoring='f1')

In [159]:
print("best parameter",rand_search.best_params_)
best_dt = rand_search.best_estimator_

best parameter {'min_samples_split': 5, 'min_samples_leaf': 4, 'max_depth': 7, 'criterion': 'gini'}


In [160]:
y_pred = best_dt.predict(X_test)
y_pred = y_pred.astype(int)

In [161]:
from sklearn.metrics import accuracy_score

accuracy_score1 = accuracy_score(y_test, y_pred)
accuracy_score1

0.415969696969697

In [162]:
import joblib

# Ensure the output directory exists
import os
output_dir = 'deployment_artifacts'
os.makedirs(output_dir, exist_ok=True)

# Save the StandardScaler
joblib.dump(sc, os.path.join(output_dir, 'standard_scaler.pkl'))

# Save the LabelEncoder (for 'gender' column)
joblib.dump(le, os.path.join(output_dir, 'label_encoder_gender.pkl'))

# Save the best Decision Tree Classifier model
joblib.dump(best_dt, os.path.join(output_dir, 'best_decision_tree_model.pkl'))

# Save the feature column names (important for deployment)
joblib.dump(X.columns.tolist(), os.path.join(output_dir, 'feature_columns.pkl'))

print(f"All deployment artifacts saved to '{output_dir}'")

All deployment artifacts saved to 'deployment_artifacts'


In [163]:
df.columns

Index(['age', 'gender', 'bmi', 'daily_steps', 'sleep_hours', 'water_intake_l',
       'calories_consumed', 'smoker', 'alcohol', 'resting_hr', 'systolic_bp',
       'diastolic_bp', 'cholesterol', 'family_history', 'disease_risk'],
      dtype='object')

In [164]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_pred = best_dt.predict(X_test)

print("Unique Predictions:")
print(np.unique(y_pred, return_counts=True))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Unique Predictions:
(array([0, 1]), array([11207, 21793]))

Confusion Matrix:
[[ 8354 16420]
 [ 2853  5373]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.34      0.46     24774
           1       0.25      0.65      0.36      8226

    accuracy                           0.42     33000
   macro avg       0.50      0.50      0.41     33000
weighted avg       0.62      0.42      0.44     33000

